In [42]:
import pandas as pd
import numpy as np

df = pd.read_csv("./iris/iris.csv")
statistical_analysis = df.describe()
df

,Id,SepalLengthCm,SepalWidthCm,PetalLengthCm,PetalWidthCm,Species
0,1,5.1,3.5,1.4,0.2,Iris-setosa
1,2,4.9,3.0,1.4,0.2,Iris-setosa
2,3,4.7,3.2,1.3,0.2,Iris-setosa
3,4,4.6,3.1,1.5,0.2,Iris-setosa
4,5,5.0,3.6,1.4,0.2,Iris-setosa
...,...,...,...,...,...,...
145,146,6.7,3.0,5.2,2.3,Iris-virginica
146,147,6.3,2.5,5.0,1.9,Iris-virginica
147,148,6.5,3.0,5.2,2.0,Iris-virginica
148,149,6.2,3.4,5.4,2.3,Iris-virginica


In [43]:
iris_setosa = df[df['Species'] == 'Iris-setosa'].drop(columns=['Species'])
iris_versicolor = df[df['Species'] == 'Iris-versicolor'].drop(columns=['Species'])
iris_virginica = df[df['Species'] == 'Iris-virginica'].drop(columns=['Species'])

In [44]:
reg_df = iris_setosa.drop(columns=["Id"])
reg_df.rename(
    columns={
        "PetalLengthCm": "petal_length",
        "PetalWidthCm": "petal_width",
        "SepalLengthCm": "sepal_length",
        "SepalWidthCm": "sepal_width",
        "Id": "id",
    },
    inplace=True,
)

training_df = reg_df[reg_df.index < reg_df.shape[0] * 0.8].reset_index(drop=True)
test_df = reg_df[reg_df.index >= reg_df.shape[0] * 0.8].reset_index(drop=True)
test_df.rename(columns={"sepal_width": "actual_sepal_width"}, inplace=True)
test_df = test_df[["sepal_length", "petal_length", "petal_width", "actual_sepal_width"]]

In [45]:
y = training_df['sepal_width'].to_numpy()
x = pd.concat([pd.DataFrame(np.ones((len(training_df), 1))), training_df['sepal_length'], training_df['petal_length'], training_df['petal_width']], axis=1).to_numpy()
x, y

(array([[1. , 5.1, 1.4, 0.2],
        [1. , 4.9, 1.4, 0.2],
        [1. , 4.7, 1.3, 0.2],
        [1. , 4.6, 1.5, 0.2],
        [1. , 5. , 1.4, 0.2],
        [1. , 5.4, 1.7, 0.4],
        [1. , 4.6, 1.4, 0.3],
        [1. , 5. , 1.5, 0.2],
        [1. , 4.4, 1.4, 0.2],
        [1. , 4.9, 1.5, 0.1],
        [1. , 5.4, 1.5, 0.2],
        [1. , 4.8, 1.6, 0.2],
        [1. , 4.8, 1.4, 0.1],
        [1. , 4.3, 1.1, 0.1],
        [1. , 5.8, 1.2, 0.2],
        [1. , 5.7, 1.5, 0.4],
        [1. , 5.4, 1.3, 0.4],
        [1. , 5.1, 1.4, 0.3],
        [1. , 5.7, 1.7, 0.3],
        [1. , 5.1, 1.5, 0.3],
        [1. , 5.4, 1.7, 0.2],
        [1. , 5.1, 1.5, 0.4],
        [1. , 4.6, 1. , 0.2],
        [1. , 5.1, 1.7, 0.5],
        [1. , 4.8, 1.9, 0.2],
        [1. , 5. , 1.6, 0.2],
        [1. , 5. , 1.6, 0.4],
        [1. , 5.2, 1.5, 0.2],
        [1. , 5.2, 1.4, 0.2],
        [1. , 4.7, 1.6, 0.2],
        [1. , 4.8, 1.6, 0.2],
        [1. , 5.4, 1.5, 0.4],
        [1. , 5.2, 1.5, 0.1],
        [1

In [46]:
x_transpose = x.T
x_multiply_x_transpose = x_transpose @ x
beta = np.linalg.inv(x_multiply_x_transpose) @ x_transpose @ y
intercept, sepal_length_coeff, petal_length_coeff, petal_width_coeff = beta
print(f"intercept: {intercept}")
print(f"sepal_length_coeff: {sepal_length_coeff}")
print(f"petal_length_coeff: {petal_length_coeff}")
print(f"petal_width_coeff: {petal_width_coeff}")

intercept: 0.15543258265690585
sepal_length_coeff: 0.7232881286978403
petal_length_coeff: -0.3461748149246333
petal_width_coeff: 0.6334801542155097


In [47]:
y_pred = test_df
prediction = intercept + sepal_length_coeff * y_pred['sepal_length'] + petal_length_coeff * y_pred['petal_length'] + petal_width_coeff * y_pred['petal_width']
final_df = pd.concat([test_df, pd.DataFrame(prediction.round(1), columns=['predicted_sepal_width'])], axis=1)
final_df

,sepal_length,petal_length,petal_width,actual_sepal_width,predicted_sepal_width
0,5.0,1.3,0.3,3.5,3.5
1,4.5,1.3,0.3,2.3,3.2
2,4.4,1.3,0.2,3.2,3.0
3,5.0,1.6,0.6,3.5,3.6
4,5.1,1.9,0.4,3.8,3.4
5,4.8,1.4,0.3,3.0,3.3
6,5.1,1.6,0.2,3.8,3.4
7,4.6,1.4,0.2,3.2,3.1
8,5.3,1.5,0.2,3.7,3.6
9,5.0,1.4,0.2,3.3,3.4


In [54]:
error = (final_df['actual_sepal_width'] - final_df['predicted_sepal_width']).abs()
error = ( error / final_df['actual_sepal_width'] ) * 100
error

0     0.000000
1    39.130435
2     6.250000
3     2.857143
4    10.526316
5    10.000000
6    10.526316
7     3.125000
8     2.702703
9     3.030303
dtype: float64